In [1]:
#pip install pymupdf
# !pip install pandas numpy nltk scikit-learn pymupdf tqdm matplotlib

In [2]:
# ======================================
# Imports
# ======================================

import re
import string
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm import tqdm

import nltk
import pymupdf

# Download NLTK resources (first time only)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User1\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
# ======================================
# Find all PDFs
# ======================================

base_path = Path("Courses")

pdf_files = list(base_path.rglob("*.pdf"))

print(f"Total PDFs: {len(pdf_files)}")

Total PDFs: 16


In [4]:
# ======================================
# Build PDF Dataset
# ======================================

dataset = []

for pdf in pdf_files:

    dataset.append(
        {
            "course": pdf.parent.name,
            "file_name": pdf.name,
            "path": str(pdf)
        }
    )

df = pd.DataFrame(dataset)

df.head()

,course,file_name,path
0,AI Tools,Lec1 CV DIP Fundamentals.pdf,Courses\AI Tools\Lec1 CV DIP Fundamentals.pdf
1,AI Tools,Lec2 CV DIP Fundamentals.pdf,Courses\AI Tools\Lec2 CV DIP Fundamentals.pdf
2,AI Tools,Lec3 CV DIP Fundamentals PI II.pdf,Courses\AI Tools\Lec3 CV DIP Fundamentals PI I...
3,AI Tools,Lec4 Convolution Neural Networks.pdf,Courses\AI Tools\Lec4 Convolution Neural Netwo...
4,AI Tools,Lec5 Naltural Language Processing.pdf,Courses\AI Tools\Lec5 Naltural Language Proces...


In [5]:
import pymupdf

def extract_pages(pdf_path):

    document = pymupdf.open(pdf_path)

    pages = []

    for page_number, page in enumerate(document, start=1):

        pages.append({
            "page": page_number,
            "text": page.get_text("text")
        })

    document.close()

    return pages

In [6]:
# ======================================
# Extract Pages
# ======================================

pages_dataset = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    pages = extract_pages(row["path"])

    for page in pages:

        pages_dataset.append(
            {
                "course": row["course"],
                "file_name": row["file_name"],
                "page": page["page"],
                "text": page["text"]
            }
        )

pages_df = pd.DataFrame(pages_dataset)

pages_df.head()

100%|██████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 24.02it/s]


,course,file_name,page,text
0,AI Tools,Lec1 CV DIP Fundamentals.pdf,1,AI Tools Emerging & Technologies\nFundamental...
1,AI Tools,Lec1 CV DIP Fundamentals.pdf,2,Computer Vision\nIs a field of AI & Computer S...
2,AI Tools,Lec1 CV DIP Fundamentals.pdf,3,Computer Graphics\nPattern Recognition\nImage ...
3,AI Tools,Lec1 CV DIP Fundamentals.pdf,4,Digital Image Processing Applications\nDigital...
4,AI Tools,Lec1 CV DIP Fundamentals.pdf,5,Digital Image Processing Applications\nDigital...


In [7]:
pages_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   course     550 non-null    object
 1   file_name  550 non-null    object
 2   page       550 non-null    int64 
 3   text       550 non-null    object
dtypes: int64(1), object(3)
memory usage: 17.3+ KB


In [8]:
# ======================================
# Text Preprocessing
# ======================================

import re
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Initialize

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

translator = str.maketrans("", "", string.punctuation)


def lowercase(text):
    return text.lower()


def remove_numbers(text):
    return re.sub(r"\d+", "", text)


def remove_punctuation(text):
    return text.translate(translator)


def tokenize(text):
    return word_tokenize(text)


def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]


def lemmatize(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]


def join_tokens(tokens):
    return " ".join(tokens)


def preprocess_text(text):

    text = lowercase(text)

    text = remove_numbers(text)

    text = remove_punctuation(text)

    tokens = tokenize(text)

    tokens = remove_stopwords(tokens)

    tokens = lemmatize(tokens)

    return join_tokens(tokens)

In [9]:
sample = pages_df.loc[0, "text"]

print(sample[:500])

AI Tools  Emerging & Technologies
Fundamentals of Computer Vision & 
Image Processing
Comp. Eng. & AI Staff              MTC, Cairo, Egypt



In [10]:
clean_text = preprocess_text(sample)

print(clean_text[:500])

ai tool emerging technology fundamental computer vision image processing comp eng ai staff mtc cairo egypt


In [11]:
from tqdm import tqdm

tqdm.pandas()

pages_df["clean_text"] = pages_df["text"].progress_apply(preprocess_text)

100%|██████████████████████████████████████████████████████████████████████████████| 550/550 [00:00<00:00, 3265.73it/s]


In [12]:
pages_df.head()

,course,file_name,page,text,clean_text
0,AI Tools,Lec1 CV DIP Fundamentals.pdf,1,AI Tools Emerging & Technologies\nFundamental...,ai tool emerging technology fundamental comput...
1,AI Tools,Lec1 CV DIP Fundamentals.pdf,2,Computer Vision\nIs a field of AI & Computer S...,computer vision field ai computer science trai...
2,AI Tools,Lec1 CV DIP Fundamentals.pdf,3,Computer Graphics\nPattern Recognition\nImage ...,computer graphic pattern recognition image pro...
3,AI Tools,Lec1 CV DIP Fundamentals.pdf,4,Digital Image Processing Applications\nDigital...,digital image processing application digital i...
4,AI Tools,Lec1 CV DIP Fundamentals.pdf,5,Digital Image Processing Applications\nDigital...,digital image processing application digital i...


In [13]:
print("Original Text\n")
print(pages_df.loc[0, "text"][:500])

print("\n" + "="*80 + "\n")

print("Clean Text\n")
print(pages_df.loc[0, "clean_text"][:500])

Original Text

AI Tools  Emerging & Technologies
Fundamentals of Computer Vision & 
Image Processing
Comp. Eng. & AI Staff              MTC, Cairo, Egypt



Clean Text

ai tool emerging technology fundamental computer vision image processing comp eng ai staff mtc cairo egypt


In [14]:
# ======================================
# Build Vocabulary
# ======================================

vocabulary = set()

for text in pages_df["clean_text"]:

    words = text.split()

    vocabulary.update(words)

vocabulary = sorted(vocabulary)

print("Vocabulary Size:", len(vocabulary))

Vocabulary Size: 3459


In [15]:
vocabulary[:50]

['ab',
 'ability',
 'able',
 'abruptly',
 'absence',
 'absolute',
 'abstract',
 'academic',
 'acceleration',
 'accelerator',
 'accelerometer',
 'access',
 'accessed',
 'accessible',
 'according',
 'account',
 'accuracy',
 'accurate',
 'accurately',
 'achieve',
 'achieved',
 'achieves',
 'acquire',
 'acquired',
 'acquisition',
 'across',
 'act',
 'acted',
 'acting',
 'action',
 'activate',
 'activated',
 'activates',
 'activation',
 'active',
 'actively',
 'activity',
 'actor',
 'actual',
 'actually',
 'actuator',
 'ad',
 'adapt',
 'adaptation',
 'adaptive',
 'adapts',
 'add',
 'added',
 'adding',
 'addition']

In [16]:
word_to_index = {}

for index, word in enumerate(vocabulary):

    word_to_index[word] = index

In [17]:
print(word_to_index["machine"])

1663


In [18]:
index_to_word = {}

for word, index in word_to_index.items():

    index_to_word[index] = word

In [19]:
print(index_to_word[1845])

mutually


In [20]:
sample = pages_df.loc[0, "clean_text"]

print(sample[:500])

ai tool emerging technology fundamental computer vision image processing comp eng ai staff mtc cairo egypt


In [21]:
tokens = sample.split()

for token in tokens[:30]:

    print(token, "---->", word_to_index[token])

ai ----> 76
tool ----> 2839
emerging ----> 843
technology ----> 2785
fundamental ----> 1118
computer ----> 494
vision ----> 3018
image ----> 1346
processing ----> 2163
comp ----> 468
eng ----> 866
ai ----> 76
staff ----> 2657
mtc ----> 1823
cairo ----> 334
egypt ----> 828


In [22]:
# ======================================
# Build Bag of Words for One Document
# ======================================

def build_bow(document, vocabulary, word_to_index):

    # إنشاء Vector كله أصفار
    bow_vector = [0] * len(vocabulary)

    # تقسيم الكلمات
    words = document.split()

    # عدّ الكلمات
    for word in words:

        if word in word_to_index:

            index = word_to_index[word]

            bow_vector[index] += 1

    return bow_vector

In [23]:
sample = pages_df.loc[0, "clean_text"]

bow = build_bow(sample, vocabulary, word_to_index)

print("Vector Length:", len(bow))
print(bow[:30])

Vector Length: 3459
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [24]:
for word in sample.split():

    index = word_to_index[word]

    print(f"{word:<20} --> {bow[index]}")

ai                   --> 2
tool                 --> 1
emerging             --> 1
technology           --> 1
fundamental          --> 1
computer             --> 1
vision               --> 1
image                --> 1
processing           --> 1
comp                 --> 1
eng                  --> 1
ai                   --> 2
staff                --> 1
mtc                  --> 1
cairo                --> 1
egypt                --> 1


In [25]:
from tqdm import tqdm

bow_matrix = []

for document in tqdm(pages_df["clean_text"]):

    bow_vector = build_bow(document, vocabulary, word_to_index)

    bow_matrix.append(bow_vector)

100%|█████████████████████████████████████████████████████████████████████████████| 550/550 [00:00<00:00, 47942.87it/s]


In [26]:
bow_matrix = np.array(bow_matrix)

print(bow_matrix.shape)

(550, 3459)


In [27]:
bow_matrix[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(3459,))

In [28]:
bow_df = pd.DataFrame(
    bow_matrix,
    columns=vocabulary
)

bow_df.head()

,ab,ability,able,abruptly,absence,absolute,abstract,academic,acceleration,accelerator,...,scraping,selftraining,semiautomatically,squad,the,there,these,they,this,web
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
print(bow_matrix.shape)

(550, 3459)


In [30]:
# ======================================
# Build TF Vector
# ======================================

def build_tf(bow_vector):

    total_words = sum(bow_vector)

    if total_words == 0:
        return bow_vector

    tf_vector = []

    for count in bow_vector:

        tf_vector.append(count / total_words)

    return tf_vector

In [31]:
tf = build_tf(bow_matrix[0])

print(tf[:20])

[np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]


In [32]:
tf_matrix = []

for bow in tqdm(bow_matrix):

    tf_matrix.append(
        build_tf(bow)
    )

tf_matrix = np.array(tf_matrix)

print(tf_matrix.shape)

100%|██████████████████████████████████████████████████████████████████████████████| 550/550 [00:00<00:00, 1778.45it/s]


(550, 3459)


In [33]:
tf_df = pd.DataFrame(
    tf_matrix,
    columns=vocabulary
)

tf_df.head()

,ab,ability,able,abruptly,absence,absolute,abstract,academic,acceleration,accelerator,...,scraping,selftraining,semiautomatically,squad,the,there,these,they,this,web
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.033333,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
# ======================================
# Document Frequency (DF)
# ======================================

document_frequency = {}

for word in vocabulary:
    document_frequency[word] = 0

for document in pages_df["clean_text"]:

    unique_words = set(document.split())

    for word in unique_words:

        document_frequency[word] += 1

In [35]:
print(document_frequency["machine"])

78


In [36]:
import math

N = len(pages_df)

idf = {}

for word in vocabulary:

    df = document_frequency[word]

    idf[word] = math.log(N / df)

In [37]:
print(idf["machine"])

1.953209451536925


In [38]:
idf_vector = []

for word in vocabulary:

    idf_vector.append(idf[word])

idf_vector = np.array(idf_vector)

print(idf_vector.shape)

(3459,)


In [39]:
# ======================================
# TF-IDF Matrix
# ======================================

tfidf_matrix = tf_matrix * idf_vector

print(tfidf_matrix.shape)

(550, 3459)


In [40]:
tfidf_df = pd.DataFrame(
    tfidf_matrix,
    columns=vocabulary
)

tfidf_df.head()

,ab,ability,able,abruptly,absence,absolute,abstract,academic,acceleration,accelerator,...,scraping,selftraining,semiautomatically,squad,the,there,these,they,this,web
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.210331,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
first_document = tfidf_df.iloc[0]

top_words = first_document.sort_values(ascending=False).head(20)

print(top_words)

comp             0.272751
cairo            0.272751
staff            0.272751
mtc              0.272751
eng              0.272751
emerging         0.272751
egypt            0.272751
fundamental      0.264405
technology       0.244501
vision           0.229429
ai               0.213094
tool             0.190739
computer         0.165397
processing       0.116704
image            0.097271
reinforcement    0.000000
relate           0.000000
related          0.000000
refines          0.000000
refining         0.000000
Name: 0, dtype: float64


In [42]:
word = "ai"

print(idf[word])

1.7047480922384253


In [43]:
word = "backpropagation"

print(idf[word])

4.518158808998462


In [44]:
# ======================================
# Convert Query to BoW
# ======================================

def query_to_bow(query, vocabulary, word_to_index):

    query = preprocess_text(query)

    bow = [0] * len(vocabulary)

    for word in query.split():

        if word in word_to_index:

            bow[word_to_index[word]] += 1

    return np.array(bow)

In [45]:
query = "What is Machine Learning?"

query_bow = query_to_bow(
    query,
    vocabulary,
    word_to_index
)

print(query_bow.shape)

(3459,)


In [46]:
query_tf = np.array(build_tf(query_bow))

print(query_tf.shape)

(3459,)


In [47]:
query_tfidf = query_tf * idf_vector

print(query_tfidf.shape)

(3459,)


In [48]:
# ======================================
# Cosine Similarity
# ======================================

def cosine_similarity(vector1, vector2):

    numerator = np.dot(vector1, vector2)

    denominator = (
        np.linalg.norm(vector1)
        *
        np.linalg.norm(vector2)
    )

    if denominator == 0:
        return 0

    return numerator / denominator

In [49]:
score = cosine_similarity(
    query_tfidf,
    tfidf_matrix[0]
)

print(score)

0.0


In [50]:
similarities = []

for document in tfidf_matrix:

    similarities.append(
        cosine_similarity(
            query_tfidf,
            document
        )
    )

similarities = np.array(similarities)

In [51]:
top_indices = similarities.argsort()[::-1][:5]

top_indices

array([354, 355, 365, 374, 363])

In [52]:
results = pages_df.iloc[top_indices][
    [
        "course",
        "file_name",
        "page"
    ]
].copy()

results["score"] = similarities[top_indices]

results

,course,file_name,page,score
354,ML,Lecture3 Introduction to ML.pdf,15,1.000000
355,ML,Lecture3 Introduction to ML.pdf,16,1.000000
365,ML,Lecture3 Introduction to ML.pdf,26,0.704833
374,ML,Lecture3 Introduction to ML.pdf,35,0.668260
363,ML,Lecture3 Introduction to ML.pdf,24,0.644788


In [53]:
for idx in top_indices:

    print("=" * 80)

    print("Course :", pages_df.loc[idx, "course"])

    print("File   :", pages_df.loc[idx, "file_name"])

    print("Page   :", pages_df.loc[idx, "page"])

    print("Score  :", round(similarities[idx], 4))

    print()

    print(pages_df.loc[idx, "text"][:1000])

    print("\n")

Course : ML
File   : Lecture3 Introduction to ML.pdf
Page   : 15
Score  : 1.0

15
Machine Learning? 



Course : ML
File   : Lecture3 Introduction to ML.pdf
Page   : 16
Score  : 1.0

16
Machine Learning? 



Course : ML
File   : Lecture3 Introduction to ML.pdf
Page   : 26
Score  : 0.7048

26
Machine Learning vs. Deep Learning



Course : ML
File   : Lecture3 Introduction to ML.pdf
Page   : 35
Score  : 0.6683

Agenda
1. Foundations of Machine Learning
2. Process of building ML-based systems
3. Machine Vs. Deep learning
4. Big Data
5.5. Types Of Machine Learning 
Types Of Machine Learning 
35



Course : ML
File   : Lecture3 Introduction to ML.pdf
Page   : 24
Score  : 0.6448

Agenda
1. Foundations of Machine Learning
2. Process of building ML-based systems
3.3. Machine Vs. Deep learning
Machine Vs. Deep learning
4. Big Data
5. Types Of Machine Learning 
24





In [54]:
import pickle
import os

os.makedirs("data", exist_ok=True)

In [55]:
with open("data/vocabulary.pkl", "wb") as f:
    pickle.dump(vocabulary, f)

In [56]:
with open("data/word_to_index.pkl", "wb") as f:
    pickle.dump(word_to_index, f)

In [57]:
with open("data/idf.pkl", "wb") as f:
    pickle.dump(idf, f)

In [58]:
with open("data/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

In [59]:
pages_df.to_pickle("data/pages_df.pkl")

In [60]:
import os

print(os.listdir("data"))

['idf.pkl', 'pages_df.pkl', 'tfidf_matrix.pkl', 'vocabulary.pkl', 'word_to_index.pkl']


In [61]:
def search(query, top_k=5):

    # Query -> BoW
    query_bow = query_to_bow(
        query,
        vocabulary,
        word_to_index
    )

    # BoW -> TF
    query_tf = np.array(build_tf(query_bow))

    # TF -> TF-IDF
    query_tfidf = query_tf * idf_vector

    similarities = []

    for document in tfidf_matrix:

        similarities.append(
            cosine_similarity(
                query_tfidf,
                document
            )
        )

    similarities = np.array(similarities)

    top_indices = similarities.argsort()[::-1][:top_k]

    results = pages_df.iloc[top_indices].copy()

    results["score"] = similarities[top_indices]

    return results[
        [
            "course",
            "file_name",
            "page",
            "score",
            "text"
        ]
    ]

In [62]:
results = search(
    "What is Machine Learning?"
)

results

,course,file_name,page,score,text
354,ML,Lecture3 Introduction to ML.pdf,15,1.000000,15\nMachine Learning? \n
355,ML,Lecture3 Introduction to ML.pdf,16,1.000000,16\nMachine Learning? \n
365,ML,Lecture3 Introduction to ML.pdf,26,0.704833,26\nMachine Learning vs. Deep Learning\n
374,ML,Lecture3 Introduction to ML.pdf,35,0.668260,Agenda\n1. Foundations of Machine Learning\n2....
363,ML,Lecture3 Introduction to ML.pdf,24,0.644788,Agenda\n1. Foundations of Machine Learning\n2....


In [63]:
results = search(
    "Explain CNN"
)

results

,course,file_name,page,score,text
218,AI Tools,Lec7 LLM vs. GAN LMS.pdf,26,0.670919,CNN Case Study\n
141,AI Tools,Lec4 Convolution Neural Networks.pdf,29,0.670919,CNN Case Study I\n
138,AI Tools,Lec4 Convolution Neural Networks.pdf,26,0.528464,Convolutional Neural Networks CNN\n
215,AI Tools,Lec7 LLM vs. GAN LMS.pdf,23,0.528464,Convolutional Neural Networks CNN\n
131,AI Tools,Lec4 Convolution Neural Networks.pdf,19,0.528464,Convolutional Neural Networks CNN\n


In [64]:
results = search(
    "What is Gradient Descent?"
)

results

,course,file_name,page,score,text
225,AI Tools,Lec7 LLM vs. GAN LMS.pdf,33,0.769989,"Optimization, Gradient Descent\n"
226,AI Tools,Lec7 LLM vs. GAN LMS.pdf,34,0.769989,"Optimization, Gradient Descent\n0\n1\n1\n1\n"
220,AI Tools,Lec7 LLM vs. GAN LMS.pdf,28,0.425566,"Optimization, Gradient Descent\nGradient Desce..."
221,AI Tools,Lec7 LLM vs. GAN LMS.pdf,29,0.415404,"Optimization, Gradient Descent\nGradient Desce..."
58,AI Tools,Lec2 CV DIP Fundamentals.pdf,29,0.391662,Intermediate gradient.\n


In [68]:
while True:

    user_query = input("Type your question about the courses (or 'exit' to quit): ")

    if user_query.lower() == "exit":
        print("Goodbye 👋")
        break

    results = search(user_query)

    display(results)

Type your question about the courses (or 'exit' to quit):  Ahmed


,course,file_name,page,score,text
549,ML,Lecture9 SVM.pdf,31,0,PRACTICAL MACHINE DEEP LEARNING\n31\nDictionar...
180,AI Tools,Lec6 LLMs.pdf,9,0,Natural Language Processing\nWhat is a Transfo...
186,AI Tools,Lec6 LLMs.pdf,15,0,Natural Language Processing\nScores = (Q × Kᵀ)...
185,AI Tools,Lec6 LLMs.pdf,14,0,Natural Language Processing\nLike a musical ch...
184,AI Tools,Lec6 LLMs.pdf,13,0,Natural Language Processing\nEach word is a po...


Type your question about the courses (or 'exit' to quit):  exit


Goodbye 👋
